# 5. Choosing model settings

**What this shows:** how the model components group XBeach's settings, and how to
configure physics, boundaries, sediment and output.

**Prerequisites:** [4. Adding forcing](04_forcing.ipynb).

**You will learn:**

- how components replace a long flat list of `params.txt` settings
- how to choose a wave model and physical processes
- how processes are switched on, customised or switched off
- which boundary, sediment and output settings most models need
- how rompy-xbeach catches invalid settings before XBeach runs

**Data used:** none.

## Setup

Every component has a `get()` method that returns the parameters it adds to
`params.txt`. This small helper prints them.

In [1]:
from pydantic import ValidationError
from rompy.logging import config as logging_config

logging_config.update(level="WARNING")  # rompy logs every step, show warnings only


def show(component):
    """Print the XBeach parameters a component writes to params.txt."""
    for key, value in component.get(destdir=None).items():
        if isinstance(value, list):
            print("\n".join(str(item) for item in value))
        else:
            print(f"{key} = {value}")

## 1. Components

XBeach reads a few hundred settings. rompy-xbeach groups them into components, each
a field of the `Config`:

| Config field | Component | Controls |
|---|---|---|
| `physics` | `Physics` | Wave model, friction, viscosity, numerics, processes |
| `flow_boundary` | `FlowBoundaryConditions` | Flow boundaries on each side of the grid |
| `tide_boundary` | `TideBoundaryConditions` | How water levels are applied at the boundaries |
| `sediment` | `Sediment` | Sediment transport, morphology, bed composition |
| `output` | `Output` | Output variables, locations and timing |
| `hotstart`, `mpi` | `Hotstart`, `Mpi` | Restarts and parallel runs |

Anything you leave unset keeps the XBeach default and is not written to `params.txt`.

## 2. Physics and the wave model

The wave model is the only required setting:

| Wave model | Resolves | Typical use |
|---|---|---|
| `Stationary` | Wave-averaged energy, no wave groups | Mild conditions, quick studies |
| `Surfbeat` | Wave groups and infragravity waves | Storm impact, dune erosion |
| `Nonh` | Individual waves (non-hydrostatic) | Swash, overtopping, lab-scale studies |

In [2]:
from rompy_xbeach.components.physics import Physics
from rompy_xbeach.components.physics.wavemodel import Surfbeat

show(Physics(wavemodel=Surfbeat()))

wavemodel = surfbeat


Wave-model options, such as the breaker formulation, live inside the wave model object,
so only options valid for that model can be set.

In [3]:
from rompy_xbeach.components.physics.friction import Manning
from rompy_xbeach.components.physics.wavemodel import Roelvink2

physics = Physics(
    wavemodel=Surfbeat(breaktype=Roelvink2(gamma=0.55)),
    bedfriction=Manning(bedfriccoef=0.02),
)
show(physics)

wavemodel = surfbeat
break = roelvink2
gamma = 0.55
bedfriction = manning
bedfriccoef = 0.02


## 3. Switching processes on, off, or customising them

Optional processes accept `True` (on with XBeach defaults), `False` (off) or a
component with custom settings. The same pattern is used for the roller, viscosity,
vegetation, wave-current interaction, wind, sediment transport and morphology.

In [4]:
from rompy_xbeach.components.physics.friction import Viscosity

show(Physics(wavemodel=Surfbeat(), viscosity=True, wci=False))

wavemodel = surfbeat
viscosity = 1
wci = 0


In [5]:
show(Physics(wavemodel=Surfbeat(), viscosity=Viscosity(smag=False, nuh=0.5)))

wavemodel = surfbeat
viscosity = 1
smag = 0
nuh = 0.5


## 4. Boundaries

`FlowBoundaryConditions` sets the flow boundary type on each side of the grid.
`TideBoundaryConditions` sets how water levels are applied. With no tide input, use
`tideloc=0` and a constant level `zs0`, otherwise XBeach expects a tide file.

In [6]:
from rompy_xbeach.components.boundary.parameters import (
    FlowBoundaryConditions,
    TideBoundaryConditions,
)

show(
    FlowBoundaryConditions(front="abs_2d", back="wall", left="neumann", right="neumann")
)

front = abs_2d
back = wall
left = neumann
right = neumann


In [7]:
show(TideBoundaryConditions(tideloc=0, zs0=0.5))

tideloc = 0
zs0 = 0.5


## 5. Sediment and morphology

Sediment transport and bed updating are off in XBeach by default. Turn them on with a
switch or a component. `morfac` speeds up bed change relative to the hydrodynamics.

In [8]:
from rompy_xbeach.components.sediment import Sediment
from rompy_xbeach.components.sediment.composition import BedComposition
from rompy_xbeach.components.sediment.morphology import Morphology

sediment = Sediment(
    sedtrans=True,
    morphology=Morphology(morfac=10.0),
    bed_composition=BedComposition(D50=0.0003, D90=0.0005),
)
show(sediment)

sedtrans = 1
morphology = 1
morfac = 10.0
D50 = 0.0003
D90 = 0.0005


## 6. Output

`Output` chooses what XBeach writes and how often. Here it asks for hourly means of
wave height and water level, and bed-level snapshots every 30 minutes. In
`params.txt` each variable list follows its count (`nmeanvar`, `nglobalvar`), which is
filled in for you.

In [9]:
from rompy_xbeach.components.output import Output

output = Output(
    meanvars=["H", "zs"],
    tintm=3600.0,
    globalvars=["zb"],
    tintg=1800.0,
)
show(output)

outputformat = netcdf
tintg = 1800.0
tintm = 3600.0
nmeanvar = 2
H
zs
nglobalvar = 1
zb


## 7. Validation

Values are checked when you create a component, so mistakes appear in Python instead
of halfway through an XBeach run. Out-of-range values raise an error:

In [10]:
try:
    Roelvink2(gamma=2.0)
except ValidationError as error:
    print(error.errors()[0]["msg"])

Input should be less than or equal to 0.9


Unknown output variables are rejected, and the error lists the valid names:

In [11]:
try:
    Output(globalvars=["H", "wave_height"])
except ValidationError as error:
    print(error.errors()[0]["msg"][:100], "...")

Input should be 'As', 'BR', 'Cdrag', 'D', 'D15', 'D50', 'D50top', 'D90', 'D90top', 'DR', 'Dc', 'Df', ...


Some combinations are valid but suspicious, and are logged as warnings. For example,
the non-hydrostatic model resolves waves itself, so the short-wave driver should be
off:

In [12]:
from rompy_xbeach.components.physics.wavemodel import Nonh

nonh = Physics(wavemodel=Nonh(), swave=True)

2026-09-28 14:15:35 [WARNING] rompy_xbeach.components.physics.physics: ! Parameter 'swave' should not be True when using Nonh wavemodel. XBeach requires swave=0 for non-hydrostatic mode.


## Summary

- Components group related settings. Unset options keep XBeach defaults.
- `Physics(wavemodel=...)` is the only required setting.
- Processes take `True`, `False` or a component with custom settings.
- Set `tideloc=0` when there is no tide forcing.

**Next:** [6. A complete storm-impact setup](06_complete_setup.ipynb).

**See also:** the full component tours in the examples:
[physics](../examples/physics.ipynb),
[sediment and morphology](../examples/sediment_and_morphology.ipynb),
[flow and tide boundaries](../examples/flow_and_tide_boundaries.ipynb) and
[output](../examples/output.ipynb).